In [19]:
import os
import re
import sys
import pandas as pd
import numpy as np

################################################################################
# 1) Define a function that does the entire "append fraction lines" process.
################################################################################


def append_fraction_lines_to_log(
    project_path,
    connect_path,
    output_restored_name="output_restored.log",
    output_plus_name="output_restored_plus.log",
):
    """
    1) Loads the .param file from the specified project_path (like 'dcdm/dcdm_baseline_filter_thres500').
    2) Instantiates the dummy CONNECT object and the real MCMC sampler.
    3) Scans all 'number_i' folders to find iteration numbers.
    4) For each iteration, calls 'dummy_connect.print_accepted_percentage(i=i, mcmc=mcmc)'
       to get the fraction message:
         "The new points added constitutes XXX% of the total accepted pool after applying the likelihood-filter."
    5) Reads 'output_restored.log' line by line, copies it verbatim to 'output_restored_plus.log'.
       Whenever we see:
         - "Beginning iteration no. i"
           we record i.
         - "The likelihood-filter accepted X/Y new points from the chains"
           we check if i>0. If yes, we append the fraction line from step (4) for iteration i.

    Args:
        project_path (str): e.g. "dcdm/dcdm_baseline_filter_thres500"
        connect_path (str): e.g. "/home/maanson/Speciale/connectv2"
        output_restored_name (str): name of the existing combined log file to read from
        output_plus_name (str): name for the new log file to be created

    Returns:
        None (writes the new file to disk)
    """

    # -------------------------------------------------------------------------
    # 1) Load the param file
    # -------------------------------------------------------------------------
    from source.default_module import Parameters

    param_file = os.path.join(connect_path, "data", project_path, "log_connect.param")
    if not os.path.isfile(param_file):
        raise FileNotFoundError(f"Param file not found: {param_file}")
    param = Parameters(param_file)

    if param is None:
        print(f"Could not load param file {param_file}. No modifications to logs.")
        return

    # -------------------------------------------------------------------------
    # 2) Construct the dummy CONNECT object and MCMC sampler
    # -------------------------------------------------------------------------
    class DummyCONNECT:
        def __init__(self, CONNECT_PATH, data_path, param):
            self.CONNECT_PATH = CONNECT_PATH
            self.data_path = data_path
            self.param = param

        def compare_dataframes(
            self,
            df1,
            df2,
            df_likelihood=None,
            df_likelihood2=None,
            comparison_type="new",
            verbose=1,
            compare_context=None,
        ):

            import pandas as pd

            """
            
            #######
            This function was initially developed for the 'plot_iterations.py' module used to analyze the iterative sampling process.
            But it can be used here as well to compare the data overlap between the final accepted data by the likelihood-filter and the percentage of new data from the chains.
            This gives information of how large percentage of the data is actually accepted, and helps us track if the likelihood-filter is too strict for the procedure to converge naturally.
            #######

            Compare two DataFrames (df1, df2) to identify samples that are 'new', 'removed', or 'common',
            while preserving one-to-one matching of duplicates. Also re-aligns likelihood data if provided.

            Parameters
            ----------
            df1 : pd.DataFrame
                The first DataFrame (e.g., the "current" iteration's data).
            df2 : pd.DataFrame
                The second DataFrame (e.g., the "previous" iteration's data).
            df_likelihood : pd.DataFrame, optional
                Likelihood rows aligned with df1 (same length, same row order as df1 BEFORE sorting).
            df_likelihood2 : pd.DataFrame, optional
                Likelihood rows aligned with df2 (same length, same row order as df2 BEFORE sorting).
            comparison_type : {'new','removed','common'}, default='new'
                - 'new': return rows in df1 that are not in df2.
                - 'removed': return rows in df2 that are not in df1.
                - 'common': return rows present in both df1 and df2.
            verbose : int, optional
                If >0, prints some debugging info.

            Returns
            -------
            matched_params : pd.DataFrame
                Subset of parameter rows that match the requested relationship,
                extracted from the correct perspective (df1 or df2, or intersection).
            matched_likelihood : pd.DataFrame or None
                Subset of likelihood rows that align with matched_params. If none given,
                returns None.
            """

            # --------------------------------------------------
            # Step 1: Validate input parameters
            # --------------------------------------------------
            valid_types = ["new", "removed", "common"]
            if comparison_type not in valid_types:
                raise ValueError(
                    f"comparison_type must be one of {valid_types}, got: {comparison_type}"
                )

            # --------------------------------------------------
            # Step 2: Handle edge cases (if df1 or df2 is empty)
            # --------------------------------------------------
            if df1 is None or df1.empty:
                if verbose > 0:
                    print(
                        f'\n[compare_dataframes] [{compare_context["context"]}] df1 ({compare_context["df1"]}) is empty; returning trivial result:\n {compare_context["msg1"]}'
                    )
                if comparison_type == "removed" and df2 is not None:
                    return df2.copy().reset_index(drop=True), df_likelihood2
                return None, None

            if df2 is None or df2.empty:
                if verbose > 0:
                    print(
                        f'\n[compare_dataframes] [{compare_context["context"]}] df2 ({compare_context["df2"]}) is empty; returning trivial result:\n {compare_context["msg2"]}'
                    )
                if comparison_type == "new":
                    return df1.copy().reset_index(drop=True), df_likelihood
                return None, None

            # --------------------------------------------------
            # Step 2b: Ensure likelihood data and dataframes match in length
            if df_likelihood is not None and len(df_likelihood) != len(df1):
                raise ValueError(
                    f'\nLength mismatch: df_likelihood ({len(df_likelihood)}) and df1 ({compare_context["df1"]}) ({len(df1)}) are not equal.'
                )
            if df_likelihood2 is not None and len(df_likelihood2) != len(df2):
                raise ValueError(
                    f'\nLength mismatch: df_likelihood2 ({len(df_likelihood2)}) and df2 ({compare_context["df2"]}) ({len(df2)}) are not equal.'
                )

            # --------------------------------------------------
            # Step 3: Preprocess df1 (current iteration's data)
            # --------------------------------------------------
            df1 = df1.copy()
            df1["_temp_idx1"] = df1.index  # Store original row index before sorting

            # Identify the relevant parameter columns (excluding helper columns)
            param_cols = [c for c in df1.columns if c not in ["_temp_idx1", "dup_id"]]

            # Sort df1 so that identical samples appear together
            df1_sorted = df1.sort_values(param_cols, kind="mergesort").reset_index(
                drop=True
            )

            # Assign a 'dup_id' to each duplicate row so they can be matched one-to-one
            df1_sorted["dup_id"] = df1_sorted.groupby(param_cols).cumcount()

            # Reorder the likelihood data to match this new sorted order
            df_likelihood_sorted = (
                df_likelihood.iloc[df1_sorted["_temp_idx1"]].reset_index(drop=True)
                if df_likelihood is not None
                else None
            )

            # --------------------------------------------------
            # Step 4: Preprocess df2 (previous iteration's data)
            # --------------------------------------------------
            df2 = df2.copy()
            df2["_temp_idx2"] = df2.index

            df2_sorted = df2.sort_values(param_cols, kind="mergesort").reset_index(
                drop=True
            )
            df2_sorted["dup_id"] = df2_sorted.groupby(param_cols).cumcount()

            df_likelihood2_sorted = (
                df_likelihood2.iloc[df2_sorted["_temp_idx2"]].reset_index(drop=True)
                if df_likelihood2 is not None
                else None
            )

            # --------------------------------------------------
            # Step 5: Perform Merge to Find Matches
            # --------------------------------------------------
            """
            We now compare df1_sorted and df2_sorted to determine which samples belong to which category:
            
            - 'new': Samples in df1 but not in df2 (found using a LEFT JOIN)
            - 'removed': Samples in df2 but not in df1 (found using a RIGHT JOIN)
            - 'common': Samples that exist in both df1 and df2 (found using an INNER JOIN)

            The 'merge' function combines both dataframes based on their common parameter columns + 'dup_id'.
            This ensures that duplicate rows match correctly and one-to-one.
            
            The 'how' parameter controls which type of comparison we perform:
            
            - 'left' (for 'new'): Keeps all rows from df1_sorted, adds matches from df2_sorted.
            - 'right' (for 'removed'): Keeps all rows from df2_sorted, adds matches from df1_sorted.
            - 'inner' (for 'common'): Keeps only rows that exist in BOTH df1_sorted and df2_sorted.

            The 'indicator=True' adds a new column `_merge`, which labels each row as:
            - 'left_only'  → Present only in df1 (new sample)
            - 'right_only' → Present only in df2 (removed sample)
            - 'both'       → Present in both (common sample)
            """
            if comparison_type == "new":
                merge_type = "left"
                indicator = True
            elif comparison_type == "removed":
                merge_type = "right"
                indicator = True
            else:  # 'common'
                merge_type = "inner"
                indicator = False

            merged = df1_sorted.merge(
                df2_sorted,
                on=param_cols + ["dup_id"],
                how=merge_type,
                indicator=indicator,
                suffixes=("_df1", "_df2"),
            )

            # --------------------------------------------------
            # Step 6: Extract the Matching Rows from the Merge
            # --------------------------------------------------
            """
            Now that we have merged df1_sorted and df2_sorted, we extract the rows based on `_merge`:

            - For 'new': We filter only rows labeled as 'left_only' (i.e., samples that appear in df1 but not df2).
            - For 'removed': We filter only rows labeled as 'right_only' (samples in df2 but not df1).
            - For 'common': We take all merged rows, since they exist in both dataframes.
            """
            if comparison_type == "new":
                matched_df = merged[merged["_merge"] == "left_only"].drop(
                    columns=["_merge"]
                )
            elif comparison_type == "removed":
                matched_df = merged[merged["_merge"] == "right_only"].drop(
                    columns=["_merge"]
                )
            else:  # 'common'
                matched_df = merged

            # Extract the original rows from df1 or df2
            matched_params = (
                df1.iloc[matched_df["_temp_idx1"]].copy()
                if comparison_type != "removed"
                else df2.iloc[matched_df["_temp_idx2"]].copy()
            )

            # Remove helper columns
            matched_params.drop(
                columns=["dup_id", "_temp_idx1", "_temp_idx2"],
                inplace=True,
                errors="ignore",
            )
            matched_params.reset_index(drop=True, inplace=True)

            # --------------------------------------------------
            # Step 7: Extract Aligned Likelihood Data
            # --------------------------------------------------
            matched_likelihood = None
            if comparison_type in ["new", "common"] and df_likelihood_sorted is not None:
                matched_likelihood = (
                    df_likelihood.iloc[matched_df["_temp_idx1"]]
                    .copy()
                    .reset_index(drop=True)
                )
            elif comparison_type == "removed" and df_likelihood2_sorted is not None:
                matched_likelihood = (
                    df_likelihood2.iloc[matched_df["_temp_idx2"]]
                    .copy()
                    .reset_index(drop=True)
                )

            return matched_params, matched_likelihood

        def load_data_file(self, file_path, verbose=1):
            """
            Load a data file that has a header line starting with '#' and returns a DataFrame.
            """
            if not os.path.isfile(file_path):
                if verbose >= 1:
                    print(f"[load_data_file] File {file_path} does not exist.", flush=True)
                return None

            header_line = None
            with open(file_path, "r") as f:
                for line in f:
                    if line.startswith("#"):
                        header_line = line.lstrip("#").strip()
                        break

            if header_line is None:
                raise ValueError(f"No header line starting with '#' found in {file_path}")

            columns = header_line.split()
            if verbose >= 3:
                print(f"[load_data_file] Columns for {file_path}: {columns}", flush=True)

            df = pd.read_csv(
                file_path,
                sep=r"\s+",
                comment="#",
                names=columns,
                index_col=False,
                dtype=np.float32,
            )

            # Optional sanity checks
            if df.empty and verbose >= 2:
                print(
                    f"[load_data_file] Warning: Loaded DataFrame from {file_path} is empty.",
                    flush=True,
                )

            return df

        # We'll define the same method you used for printing fraction:
        def print_accepted_percentage(self, i, mcmc):
            # Reuse the logic from your snippet:
            # ----------------------
            all_accepted_after_lklfilter_path = os.path.join(
                self.CONNECT_PATH, self.data_path, f"number_{i}", "model_params.txt"
            )
            if not os.path.isfile(all_accepted_after_lklfilter_path):
                # No iteration data found
                return None

            # load_data_file
            all_accepted_after_lklfilter_df = self.load_data_file(
             all_accepted_after_lklfilter_path, verbose=0
            )
            if (
                all_accepted_after_lklfilter_df is None
                or all_accepted_after_lklfilter_df.empty
            ):
                return None

            N_all_accepted_after_lkl_filter = len(all_accepted_after_lklfilter_df)

            # Same for likelihood_data
            lkl_data_path = os.path.join(
                self.CONNECT_PATH, self.data_path, f"number_{i}", "likelihood_data.txt"
            )
            all_accepted_after_lklfilter_likelihood_df = self.load_data_file(
             lkl_data_path, verbose=0
            )

            # load chain data from MCMC
            accepted_by_oversampling = mcmc.import_points_from_chains(i)
            if accepted_by_oversampling is None or len(accepted_by_oversampling) == 0:
                return None

            # Build a small DF for oversampling
            param_cols = all_accepted_after_lklfilter_df.columns
            accepted_by_oversampling_df = pd.DataFrame(
                accepted_by_oversampling, columns=param_cols
            )

            # Compare them with compare_dataframes(...) to find how many new points survived
            compare_context = {
                "context": f"overlap iteration {i}",
                "df1": "all_accepted_after_lklfilter",
                "df2": "accepted_by_oversampling",
                "msg1": "all_accepted_after_lklfilter is empty",
                "msg2": "accepted_by_oversampling is empty",
            }
            new_accepted_df, _ = self.compare_dataframes(
                df1=all_accepted_after_lklfilter_df,
                df2=accepted_by_oversampling_df,
                df_likelihood=all_accepted_after_lklfilter_likelihood_df,
                comparison_type="common",
                compare_context=compare_context,
                verbose=0,
            )

            N_new_accepted_after_lkl_filter = len(new_accepted_df)
            # Avoid division by zero
            if N_all_accepted_after_lkl_filter == 0:
                return None

            fraction = (
                100.0
                * N_new_accepted_after_lkl_filter
                / N_all_accepted_after_lkl_filter
            )
            message = f"    The new points added constitutes {fraction:.1f}% of the total accepted pool after applying the likelihood-filter."
            return message

    # Actually create the dummy object
    dummy_connect = DummyCONNECT(
        CONNECT_PATH=connect_path, data_path=os.path.join("data", project_path), param=param
    )

    # Instantiate real MCMC sampler
    # (We do this exactly as your code snippet does)
    sampler_name = param.mcmc_sampler
    exec(
        f"from source.mcmc_samplers.{sampler_name} import {sampler_name}",
        globals(),
    )
    _locals = {"param": param, "connect_path": connect_path}
    exec(f"mcmc = {sampler_name}(param, r'{connect_path}')", globals(), _locals)
    mcmc = _locals["mcmc"]

    # -------------------------------------------------------------------------
    # 3) Identify all iteration directories (number_i) to know available i's
    # -------------------------------------------------------------------------
    iteration_dirs = []
    data_folder = os.path.join(connect_path,"data", project_path)
    for entry in os.listdir(data_folder):
        if entry.startswith("number_"):
            try:
                i = int(entry.split("_")[1])
                iteration_dirs.append(i)
            except:
                pass
    iteration_dirs.sort()
    if not iteration_dirs:
        print(f"No iteration folders found in {data_folder}. No modifications to logs.")
        return

    # -------------------------------------------------------------------------
    # 4) Build a dictionary of i -> fraction message
    # -------------------------------------------------------------------------
    messages = {}
    for i in iteration_dirs:
        # We only want fraction if i>0, but let's do it anyway:
        msg = dummy_connect.print_accepted_percentage(i=i, mcmc=mcmc)
        if msg is not None:
            messages[i] = msg

    print(f"Found fraction messages for iterations:")
    for i, msg in messages.items():
        print(f"  {i}: {msg}")

    # -------------------------------------------------------------------------
    # 5) Insert the fraction line into output_restored.log
    #    => output_restored_plus.log
    # -------------------------------------------------------------------------
    input_log = os.path.join(connect_path,"data", project_path, output_restored_name)
    output_log = os.path.join(connect_path,"data", project_path, output_plus_name)

    if not os.path.isfile(input_log):
        print(f"WARNING: Could not find {input_log}, so no changes were made.")
        return

    # Optional: remove ANSI color codes from lines before searching
    ansi_escape = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")

    current_iter = None

    # We'll read line by line from input_log, copy to output_log
    # If we see "Beginning iteration no. i", store that i in current_iter
    # If we see "The likelihood-filter accepted X/Y new points from the chains"
    # and current_iter>0, we append the messages[current_iter] if it exists.
    with open(input_log, "r", encoding="utf-8") as infile, open(
        output_log, "w", encoding="utf-8"
    ) as outfile:
        for line in infile:
            outfile.write(line)  # write line verbatim
            clean_line = ansi_escape.sub("", line)

            # detect iteration
            m_iter = re.search(r"Beginning iteration no\.\s+(\d+)", clean_line)
            if m_iter:
                current_iter = int(m_iter.group(1))
                print(f"Found iteration {current_iter}")

            # detect "The likelihood-filter accepted X/Y new points..."
            # e.g. "The likelihood-filter accepted 263/18701 new points from the chains"
            # ensure we have an iteration and we have a fraction message for it
            if current_iter is not None and current_iter > 0:
                m_lkl = re.search(
                    r"The likelihood-filter accepted (\d+)/(\d+) new points from the chains",
                    clean_line,
                )
                if m_lkl:
                    print(f"Found likelihood-filter line for iteration {current_iter}")
                    # If we have a message for this iteration, insert it
                    if current_iter in messages:
                        outfile.write(messages[current_iter] + "\n")
                        print(f"Inserted fraction line for iteration {current_iter} : {messages[current_iter]}")

    print(f"Done! Created {output_log} with fraction lines inserted.")


################################################################################
# 2) Example usage in Jupyter:
################################################################################

# You can now just call e.g.:
# append_fraction_lines_to_log(
#     project_path = "dcdm/dcdm_baseline_filter_thres500",
#     connect_path = "/home/maanson/Speciale/connectv2"
# )
#
# And it will produce a new "output_restored_plus.log" in that same folder
# with your extra fraction lines appended.
#
# Note: This code depends on the functions in your cell (like compare_dataframes,
# load_data_file) – so ensure you have them in the same scope or you can inline them
# inside this function if you need to be 100% self-contained.


append_fraction_lines_to_log(
    project_path="dcdm/dcdm_baseline_filter_thres10000",
    connect_path="/home/maanson/Speciale/connectv2",
)

Found fraction messages for iterations:
  1:     The new points added constitutes 90.9% of the total accepted pool after applying the likelihood-filter.
  2:     The new points added constitutes 66.7% of the total accepted pool after applying the likelihood-filter.
  3:     The new points added constitutes 42.0% of the total accepted pool after applying the likelihood-filter.
  4:     The new points added constitutes 11.0% of the total accepted pool after applying the likelihood-filter.
  5:     The new points added constitutes 45.4% of the total accepted pool after applying the likelihood-filter.
  6:     The new points added constitutes 39.6% of the total accepted pool after applying the likelihood-filter.
  7:     The new points added constitutes 15.3% of the total accepted pool after applying the likelihood-filter.
  8:     The new points added constitutes 12.4% of the total accepted pool after applying the likelihood-filter.
  9:     The new points added constitutes 6.1% of the to

In [28]:
import os
import sys


in_dir = "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/model_params_data/model_params_4.txt"

out_dirs_Cl = [
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Cl_ee_data/Cl_ee_data_4.txt",
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Cl_tt_data/Cl_tt_data_4.txt",
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Cl_te_data/Cl_te_data_4.txt",
]

out_dirs_Pk = [
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Pk_pk_data/Pk_pk_data_4.txt",
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Pk_pk_cb_data/Pk_pk_cb_data_4.txt",
]

out_dirs_bg = [
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/bg_ang.diam.dist._data/ang.diam.dist._data_4.txt",
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/bg_conf. time [Mpc]_data/conf. time [Mpc]_data_4.txt",
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/bg_H [1\Mpc]_data/H [1\Mpc]_data_4.txt",
]

out_dirs_th = [
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/th_tau_d_data/tau_d_data_4.txt",
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/th_w_b_data/w_b_data_4.txt",
]

out_dir_derived = "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/derived_data/derived_data_4.txt"

out_dirs_ex = [
    "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/extra_rs_drag_data/rs_drag_data_4.txt",
]

out_dirs_loglkl = "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/likelihood_data/likelihood_data_4.txt"


# create param object based on /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/log_connect.param

from source.default_module import Parameters

param_file = "/home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/log_connect.param"
param = Parameters(param_file)


def final_sync():
    """
    Final sync to ensure all data is written to disk.

    This function uses the local file path variables (in_dir, out_dirs_Cl, out_dirs_Pk,
    out_dir_derived, etc.) from the parent scope. It opens each file in read-write mode,
    does a flush and fsync, and closes. No new data is written.
    """

    # Collect all possible file paths into one list:
    # NOTE: Make sure these variables (in_dir, out_dirs_Cl, etc.) are defined in this scope!
    file_paths = []

    # Always sync model_params:
    if in_dir:
        file_paths.append(in_dir)

    # Cℓ files
    if len(param.output_Cl) > 0:
        file_paths.extend(out_dirs_Cl)
    # P(k) files
    if len(param.output_Pk) > 0:
        file_paths.extend(out_dirs_Pk)
    # Background, thermodynamics
    if len(param.output_bg) > 0:
        file_paths.extend(out_dirs_bg)
    if len(param.output_th) > 0:
        file_paths.extend(out_dirs_th)
    # Possibly derived:
    if len(param.output_derived) > 0:
        file_paths.append(out_dir_derived)

    # Extra outputs
    if len(param.extra_output) > 0:
        file_paths.extend(out_dirs_ex)

    # If using likelihood filter, also sync likelihood_data:
    if param.use_likelihood_filter and param.sampling == "iterative":
        file_paths.append(out_dirs_loglkl)

    # Determine the number of samples in each file
    def count_samples(fname):
        try:
            with open(fname, "r") as f:
                lines = f.readlines()
            # Determine header offset (if the first line starts with '#')
            header = 1 if lines and lines[0].startswith("#") else 0

            # Now, based on file type, decide how many lines per sample.
            base = os.path.basename(fname)
            if "model_params" in fname:
                return len(lines) - header
            elif "derived_data" in fname:
                return len(lines) - header
            elif "likelihood_data" in fname:
                return len(lines) - header
            elif "extra_" in fname:
                return len(lines) - header
            elif "/Cl_" in fname:
                return (len(lines) - header) // 2
            elif "/bg_" in fname:
                return (len(lines) - header) // 2
            elif "/th_" in fname:
                return (len(lines) - header) // 2
            elif "/Pk_" in fname:
                return (len(lines) - header) // (1 + len(param.z_Pk_list))
            else:
                return None
        except Exception as e:
            print(f"Error counting samples in {fname}: {repr(e)}", file=sys.stderr)
            return None

    # Collect sample counts for files we can interpret
    sample_counts = []
    for fname in file_paths:
        cnt = count_samples(fname)
        if cnt is not None:
            sample_counts.append(cnt)
            print(f"Found {cnt} samples in {fname}")
    if not sample_counts:
        print(
            "No valid files to check sample counts. Skipping consistency check.",
            file=sys.stderr,
        )
        return

    # Determine the minimum sample count across files
    min_samples = min(sample_counts)

    # Now, for each file, compute its expected line count and truncate if necessary.


    def expected_lines(fname):
        try:
            with open(fname, "r") as f:
                first_line = f.readline()
            header = 1 if first_line.startswith("#") else 0

            # Check the full file path for identifying substrings
            if "model_params" in fname:
                return min_samples + header
            if "derived_data" in fname:
                return min_samples + header
            if "likelihood_data" in fname:
                return min_samples + header
            if "extra_" in fname:
                return min_samples + header
            if "/Cl_" in fname:
                return 2 * min_samples + header
            if "/bg_" in fname:
                return 2 * min_samples + header
            if "/th_" in fname:
                return 2 * min_samples + header
            if "/Pk_" in fname:
                return (1 + len(param.z_Pk_list)) * min_samples + header
            else:
                return None
        except Exception as e:
            print(f"Error computing expected lines for {fname}: {repr(e)}", file=sys.stderr)
            return None

    # Truncate files that have extra lines
    for fname in file_paths:
        expected = expected_lines(fname)
        if expected is None:
            print(
                f"coudldn't determine expected lines for {fname}. Skipping truncation for all files.",
                file=sys.stderr,
            )
            return
        try:
            with open(fname, "r") as f:
                lines = f.readlines()
        except Exception as e:
            print(
                f"Error reading file {fname} during truncation: {repr(e)}",
                file=sys.stderr,
            )
            continue
        current = len(lines)
        if current > expected:
            print(
                f"[FINAL_SYNC] Truncating {fname} from {current} to {expected} lines",
                file=sys.stderr,
            )
            try:
                with open(fname, "w") as f:
                    f.writelines(lines[:expected])
            except Exception as e:
                print(f"Error truncating file {fname}: {repr(e)}", file=sys.stderr)
        elif current < expected:
            print(
                f"[FINAL_SYNC] Warning: {fname} has only {current} lines but expected {expected} lines",
                file=sys.stderr,
            )


final_sync()

Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/model_params_data/model_params_4.txt
Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Cl_ee_data/Cl_ee_data_4.txt
Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Cl_tt_data/Cl_tt_data_4.txt
Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Cl_te_data/Cl_te_data_4.txt
Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Pk_pk_data/Pk_pk_data_4.txt
Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_tests/lcdm_likelihoodfilter_discard_worst_first_True/number_9/Pk_pk_cb_data/Pk_pk_cb_data_4.txt
Found 35 samples in /home/maanson/Speciale/connectv2/data/lcdm_test